**Data Info**
- All 15-minute field audio recordings are processed using BirdNET with confidence level >= 0.1
- Results for each Audiomoth are stored in tables named "BirdNET_CombinedTable.csv" which are subsequently renamed according to the Audiomoth's identifier, f.e. "ps1.csv"
- All renamed files are consolidated into a single folder for further processing.


In [1]:
import os
import pandas as pd


Defining functions

In [2]:
def process_file(file_path, folder):
    '''
    Process a single .csv file to extract features and save it as '_processed.csv' in the same folder
    '''
    df = pd.read_csv(file_path)
    
    # extract file name and remove the extension
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    df['ID'] = file_name

    # extract date and time features from the 'File' column using regex:
    # (\d{8}) captures 8 digits (date)
    # _ matches the underscore
    # (\d{6}) captures 6 digits (time)
    df[['Date_full', 'Time_full']] = df['File'].str.extract(r'(\d{8})_(\d{6})')
    
    df['Year'] = df['Date_full'].str[:4]
    df['Month'] = df['Date_full'].str[4:6]
    df['Day'] = df['Date_full'].str[6:]
    df['Time'] = df['Time_full'].str[:2]
    
    df.drop(columns=['Date_full', 'Time_full', 'File'], inplace=True)
    
    # define the output file path (saved in the same folder with a modified name)
    output_file_path = os.path.join(folder, f'{file_name}_processed.csv')

    df.to_csv(output_file_path, index=False)
    
    return output_file_path



In [4]:
def merge_processed_files(folder, merged_file='merged_processed.csv'):
    '''
    Merge all processed .csv files in the specified folder into a single .csv file
    and save it as 'merged_processed.csv' in the same folder.

    '''
    dataframes = []
    
    # iterate over the folder to find all files ending with '_processed.csv'
    for file_name in os.listdir(folder):
        if file_name.endswith('_processed.csv'):
            file_path = os.path.join(folder, file_name)
            df = pd.read_csv(file_path)
            dataframes.append(df)
    
    if dataframes:
        merged_df = pd.concat(dataframes, ignore_index=True)
        merged_file_path = os.path.join(folder, merged_file)
        merged_df.to_csv(merged_file_path, index=False)

        return merged_df

    return pd.DataFrame()


Processings and merging all .csv files

In [ ]:
folder = 'birdnet_data/field'

for file_name in os.listdir(folder):
    if file_name.endswith('.csv') and not file_name.endswith('_processed.csv'):
        file_path = os.path.join(folder, file_name)
        process_file(file_path, folder)

merge_processed_files(folder)


,Start (s),End (s),Scientific name,Common name,Confidence,ID,Year,Month,Day,Time
0,27.0,30.0,Brachyramphus marmoratus,Marbled Murrelet,0.1457,psh1,2024,6,3,4
1,300.0,303.0,Brachyramphus marmoratus,Marbled Murrelet,0.4102,psh1,2024,6,3,4
2,321.0,324.0,Lullula arborea,Wood Lark,0.1571,psh1,2024,6,3,4
3,321.0,324.0,Loxioides bailleui,Palila,0.1168,psh1,2024,6,3,4
4,426.0,429.0,Brachyramphus marmoratus,Marbled Murrelet,0.1215,psh1,2024,6,3,4
...,...,...,...,...,...,...,...,...,...,...
826589,603.0,606.0,Carpodacus erythrinus,Common Rosefinch,0.2271,psm9,2024,7,26,7
826590,786.0,789.0,Phylloscopus nitidus,Green Warbler,0.7965,psm9,2024,7,26,7
826591,786.0,789.0,Phylloscopus trochiloides,Greenish Warbler,0.1199,psm9,2024,7,26,7
826592,801.0,804.0,Phylloscopus trochiloides,Greenish Warbler,0.3899,psm9,2024,7,26,7


Merging birdnet dataset with audiomoth locations' dataset

In [10]:
df = pd.read_csv('birdnet_data/field/merged_processed.csv')
df2 = pd.read_excel('maps/loc_audio.xlsx')

df_merged = pd.merge(df, df2, on='ID')
df_merged.head()

,Start (s),End (s),Scientific name,Common name,Confidence,ID,Year,Month,Day,Time,Latitude,Longitude,Elevation
0,27.0,30.0,Brachyramphus marmoratus,Marbled Murrelet,0.1457,psh1,2024,6,3,4,33.135856,76.45039,3644
1,300.0,303.0,Brachyramphus marmoratus,Marbled Murrelet,0.4102,psh1,2024,6,3,4,33.135856,76.45039,3644
2,321.0,324.0,Lullula arborea,Wood Lark,0.1571,psh1,2024,6,3,4,33.135856,76.45039,3644
3,321.0,324.0,Loxioides bailleui,Palila,0.1168,psh1,2024,6,3,4,33.135856,76.45039,3644
4,426.0,429.0,Brachyramphus marmoratus,Marbled Murrelet,0.1215,psh1,2024,6,3,4,33.135856,76.45039,3644


Combining data with the list of all the species that are commonly observed in the field area.  
Adding a new column 'Habitant" that indicates whether each identified species belongs to the list provided by the domain expert.  
1 = belongs to the list, 0 = does not belong.

In [11]:
df_all = pd.read_excel('birdnet_data/species/all_species_20250330.xlsx')

df_merged['Scientific name'] = df_merged['Scientific name'].str.lower()
df_merged['Common name'] = df_merged['Common name'].str.lower()

df_all['Scientific Name'] = df_all['Scientific Name'].str.lower()
df_all['Species Name'] = df_all['Species Name'].str.lower()

df_merged['Habitant'] = df_merged['Scientific name'].isin(df_all['Scientific Name']).astype(int)

df_merged.head()

,Start (s),End (s),Scientific name,Common name,Confidence,ID,Year,Month,Day,Time,Latitude,Longitude,Elevation,Habitant
0,27.0,30.0,brachyramphus marmoratus,marbled murrelet,0.1457,psh1,2024,6,3,4,33.135856,76.45039,3644,0
1,300.0,303.0,brachyramphus marmoratus,marbled murrelet,0.4102,psh1,2024,6,3,4,33.135856,76.45039,3644,0
2,321.0,324.0,lullula arborea,wood lark,0.1571,psh1,2024,6,3,4,33.135856,76.45039,3644,0
3,321.0,324.0,loxioides bailleui,palila,0.1168,psh1,2024,6,3,4,33.135856,76.45039,3644,0
4,426.0,429.0,brachyramphus marmoratus,marbled murrelet,0.1215,psh1,2024,6,3,4,33.135856,76.45039,3644,0


In [12]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 770144 entries, 0 to 770143
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Start (s)        770144 non-null  float64
 1   End (s)          770144 non-null  float64
 2   Scientific name  770144 non-null  object 
 3   Common name      770144 non-null  object 
 4   Confidence       770144 non-null  float64
 5   ID               770144 non-null  object 
 6   Year             770144 non-null  int64  
 7   Month            770144 non-null  int64  
 8   Day              770144 non-null  int64  
 9   Time             770144 non-null  int64  
 10  Latitude         770144 non-null  float64
 11  Longitude        770144 non-null  float64
 12  Elevation        770144 non-null  int64  
 13  Habitant         770144 non-null  int32  
dtypes: float64(5), int32(1), int64(5), object(3)
memory usage: 79.3+ MB


In [13]:
df_merged.duplicated().sum()

0

In [ ]:
df_merged.to_csv('data/data_20250515.csv')